# 🔍 Explorateur de Schéma — Analyse Safe (Sans données sensibles)
---
> Ce notebook extrait uniquement la **structure** du fichier, sans exposer de données réelles.

In [ ]:
import pandas as pd
import numpy as np

# ✅ MODIFIE UNIQUEMENT CETTE LIGNE
FICHIER = 'ton_fichier.csv'  # <-- mets le nom de ton fichier ici

df = pd.read_csv(FICHIER)
print(f'✅ Fichier chargé avec succès : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

## 📐 1. Dimensions du fichier

In [ ]:
dimensions = pd.DataFrame({
    'Métrique': ['Nombre de lignes', 'Nombre de colonnes', 'Taille mémoire'],
    'Valeur': [
        f"{df.shape[0]:,}",
        f"{df.shape[1]}",
        f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
    ]
})
dimensions.style.set_properties(**{'text-align': 'left'}).hide(axis='index')

## 🗂️ 2. Schéma des colonnes

In [ ]:
schema = pd.DataFrame({
    'Colonne'           : df.columns,
    'Type'              : df.dtypes.values,
    'Valeurs Nulles'    : df.isnull().sum().values,
    'Nulls (%)'         : (df.isnull().mean() * 100).round(2).values,
    'Valeurs Uniques'   : df.nunique().values,
    'Exemple (masqué)'  : [str(df[col].dropna().iloc[0])[:4] + '****' 
                           if len(df[col].dropna()) > 0 else 'N/A' 
                           for col in df.columns]
})

schema.style\
    .background_gradient(subset=['Nulls (%)'], cmap='Reds')\
    .background_gradient(subset=['Valeurs Uniques'], cmap='Blues')\
    .hide(axis='index')

## 📊 3. Statistiques des colonnes numériques

In [ ]:
stats_num = df.describe().T.round(2)
stats_num.index.name = 'Colonne'
stats_num.reset_index(inplace=True)

stats_num.style\
    .background_gradient(subset=['mean'], cmap='coolwarm')\
    .background_gradient(subset=['std'], cmap='Oranges')\
    .hide(axis='index')

## 🔤 4. Statistiques des colonnes texte / catégorielles

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns

if len(cat_cols) > 0:
    stats_cat = pd.DataFrame({
        'Colonne'          : cat_cols,
        'Valeurs Uniques'  : [df[col].nunique() for col in cat_cols],
        'Valeur la + fréquente' : [df[col].value_counts().index[0] if df[col].nunique() < 20 
                                   else '(trop de valeurs)' for col in cat_cols],
        'Fréquence (%)'    : [round(df[col].value_counts().iloc[0] / len(df) * 100, 2) 
                              if df[col].nunique() < 20 else '-' for col in cat_cols]
    })
    stats_cat.style.hide(axis='index')
else:
    print('Aucune colonne catégorielle détectée.')

## ⚠️ 5. Qualité des données — Rapport de santé

In [ ]:
total_cells = df.shape[0] * df.shape[1]
total_nulls = df.isnull().sum().sum()
doublons    = df.duplicated().sum()

rapport = pd.DataFrame({
    'Indicateur': [
        'Cellules totales',
        'Cellules vides',
        'Taux de complétude',
        'Lignes dupliquées',
        'Colonnes avec nulls > 30%'
    ],
    'Valeur': [
        f"{total_cells:,}",
        f"{total_nulls:,}",
        f"{round((1 - total_nulls/total_cells)*100, 2)} %",
        f"{doublons:,}",
        f"{(df.isnull().mean() > 0.3).sum()}"
    ]
})

rapport.style.hide(axis='index')

## 📤 6. Export du schéma (à partager en toute sécurité)

In [ ]:
schema.to_csv('schema_export.csv', index=False)
stats_num.to_csv('stats_numeriques.csv', index=False)
print('✅ Fichiers exportés : schema_export.csv & stats_numeriques.csv')
print('🔒 Ces fichiers ne contiennent aucune donnée sensible — safe à partager.')